
# Scripted Trade with Heston

Demonstrate using the Heston model 
- calibration to DAX, STOXX50 and S&P500 options
- pricing: Vanilla Equity Option, Barrier Option, Rainbow Option

### Modules

In [ ]:
from ORE import *
import sys, time, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.append('..')
import utilities
import cantaro

### Run with Black model

In [ ]:
params = Parameters()
params.fromFile("Input/ore_black.xml")
ore = OREApp(params)
ore.run()
utilities.checkErrorsAndRunTime(ore)

### NPV report

In [ ]:
reportBlack = ore.getReport("npv")
utilities.format_report(reportBlack)

### Run again with Heston calibration

In [ ]:
params = Parameters()
params.fromFile("Input/ore_heston.xml")
ore = OREApp(params)
ore.run()
utilities.checkErrorsAndRunTime(ore)
reportHeston = ore.getReport("npv")
utilities.format_report(reportHeston)

### Calibration reports

In [ ]:
import pandas as pd

df1 = pd.read_csv('Output/assetmodel_calibration.csv')
display(df1)

df2 = pd.read_csv('Output/assetmodel_calibration_detail.csv')
display(df2)

### Filter one index and expiry

In [ ]:
trade = "RainbowOption"
index = "EQ-RIC:.STOXX50E"
#index = "EQ-RIC:.GDAXI"
#index = "EQ-RIC:.SPX"
expiry = "3M"

df1 = df1[df1["#TradeID"] == trade]
df1 = df1[df1["Index"] == index]
display(df1)

df2 = df2[df2["#TradeID"] == trade]
df2 = df2[df2["Index"] == index]
df2 = df2[df2["Expiry"] == expiry]
display(df2)

### Plot smile section

In [ ]:
x = df2["Moneyness"]
y1 = df2["MarketVol"]
y2 = df2["ModelVol"]

plt.plot(x, y1, marker="o", linewidth=2, label='Market Vol')
plt.plot(x, y2, marker="3", linewidth=2, label='Model Vol')

plt.xlabel("Moneyness") 
plt.ylabel("Implied Vol") 
plt.title(index + ' Expiry ' + expiry)
plt.legend()

plt.show()

### Get some additional results

In [ ]:
res = pd.read_csv('Output/additional_results.csv')

# filter the trade
res = res[res["#TradeId"] == trade]
#display(res[res["ResultId"].str.contains("Heston")])
#display(res[res["ResultId"].str.contains("Heston.Index")])
display(res[res["ResultId"].str.contains("Heston.Forward")])


### Create QuantLib Heston Process

... to utilise the Heston PDF 

In [ ]:
spot = 4337.500000
forward = 4371.906927 
time = 0.25
v0 = 0.024300
kappa = 13.977853
theta = 0.032730
sigma = 1.912942 
rho = -0.638495

zeroYield = 0.0
drift = np.log(forward/spot)/time
dividendYield =  drift + zeroYield

Settings.instance().evaluationdate = Date(5, June, 2023)
dayCount = ActualActual(ActualActual.ISDA)
calendar = TARGET()
spotHandle = QuoteHandle(SimpleQuote(spot))
yieldTS = YieldTermStructureHandle(FlatForward(0, calendar, zeroYield, dayCount))
dividendTS = YieldTermStructureHandle(FlatForward(0, calendar, dividendYield, dayCount))
discretization = HestonProcess.QuadraticExponential
process =  HestonProcess(yieldTS, dividendTS, spotHandle, v0, kappa, theta, sigma, rho, discretization)

print("feller:  ", 2*theta*kappa/sigma**2)
print("spot:    ", spot)
print("forward: ", spotHandle.value() * yieldTS.discount(time) / dividendTS.discount(time), forward)
print("ln(spot):", np.log(spot))

# what's wrong with QuantLib's Heston pdf?
# use cantaro86 below instead

### Get path data 

In [ ]:
np.set_printoptions(linewidth=np.inf)

date = "2023-09-05"

res = pd.read_csv('Output/assetmodel_paths.csv')
res = res[res["#TradeId"] == trade]
res = res[res["Index"] == index]
res = res[res["Date"] == date]
# extract the sample values
data = res["Value"].to_numpy(dtype="float32")

# convert to log ( s(t) / spot )
data = np.log(data/spot)

#print(res)
print ("data: ", data)
print("length:", len(data))
print("mean:  ", np.mean(data))
print("stdev: ", math.sqrt(np.var(data)))
      

In [ ]:
x = np.linspace(-0.5, 0.3, 200)

plt.hist(data, density=True, bins=100, label="simulation")
plt.plot(x, [cantaro.Heston_pdf(y, t=time, v0=v0, mu=drift, theta=theta, sigma=sigma, kappa=kappa, rho=rho) for y in x], 
         linewidth=3, label="semi-analytic")
plt.xlabel("log(S/S0)") 
plt.ylabel("Density") 
plt.title("")
plt.xlim(-0.5, 0.3)
plt.legend(loc="upper left")
plt.show()